# Unidad 33 — Retroceso (backtracking)

Semana 5 · Clase 1 · Fuente: PDF parte 1, págs. 39–84

[Índice de la semana](../semana-05.ipynb) | [Anterior](u32-programacion-dinamica.ipynb) | [Siguiente](../clase-2-modulos-y-pandas/u34-modulos-de-python.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## Objetivos de la unidad

- Entender la técnica de retroceso (*backtracking*) y resolver el problema de las N reinas con ella.
- Entender que el retroceso es una búsqueda en profundidad que se apoya en la pila de llamadas de las funciones.
- Implementar algoritmos de forma eficaz mediante la poda durante la búsqueda en profundidad.

Visión general del aprendizaje: resolver el problema de las N reinas, conocer el principio de resolución de la suma de subconjuntos y aplicar técnicas de retroceso. Palabras clave: retroceso, prometedor (*promising*), búsqueda, profunda, poda.

**Nota sobre el PDF:** las diapositivas de objetivos (págs. 40–42) llevan en el encabezado la etiqueta "UNIDAD 32", pero su contenido es el de la Unidad 33 (Retroceso). Es un error de rotulación del PDF. Además, el PDF anuncia la suma de subconjuntos como parte de la visión general, pero no la desarrolla en las diapositivas: aquí se incluye como ampliación.

**Nota sobre el PDF:** en algunos renderizados de las diapositivas los títulos y los párrafos de texto aparecen distorsionados (letras recortadas), aunque el texto extraído del archivo está completo y el código y los diagramas se ven bien. Todo el contenido de esta unidad se contrastó con el texto extraído y con las imágenes de las páginas de código y de los árboles.

**Referencia:** PDF parte 1, págs. 39–42

## Contexto

El rompecabezas de las 8 reinas consiste en colocar ocho reinas en un tablero de ajedrez de 8*8. En el ajedrez la reina se mueve a izquierda y derecha, arriba y abajo, y en diagonal; por lo tanto, no puede haber otra reina en la misma fila, columna o diagonal que una reina ya colocada.

Para generalizarlo se habla del **problema de las N reinas**: colocar N reinas en un tablero N*N sin que ninguna ataque a otra. Por ejemplo, con N=4 son cuatro reinas en un tablero de 4*4. La misión de la unidad es construir un algoritmo que resuelva el problema para cualquier N.

**Referencia:** PDF parte 1, págs. 44–46

## Búsqueda en profundidad (DFS)

Antes del retroceso hay que recordar cómo se recorre un árbol. En la **búsqueda en profundidad** (*depth first search*, DFS) se prioriza la profundidad: se visita la raíz, luego uno de sus hijos, luego un hijo de este, y así hasta llegar a un nodo sin hijos por visitar. Entonces se vuelve (*backtrack*) al nodo anterior y, si le quedan hijos sin visitar, se continúa por ahí.

En el árbol del PDF el orden de visita es: raíz 1; nodo 2 y su hijo 3; al agotarse el 3 se retrocede al 2 y se visitan 4 y 5; al agotarse todos los hijos del 2 se retrocede al 1 y se sigue con el 6 (hijos 7 y 8) y con el 9 (hijos 10, 11 y 12). La numeración de cada nodo es el orden en que lo visita la DFS.

El pseudocódigo del PDF es recursivo: la propia **pila de llamadas** guarda el camino recorrido, y "retroceder" es simplemente terminar la llamada actual y volver a la anterior.

```
def depth_first_search(node v):
    visit v
    for each child u of v:
        depth_first_search(u)
```

**Referencia:** PDF parte 1, págs. 49–54

In [2]:
# El árbol del PDF (págs. 53–54), como diccionario: nodo -> lista de hijos
arbol = {
    1: [2, 6, 9],
    2: [3, 4, 5],
    6: [7, 8],
    9: [10, 11, 12],
}

def depth_first_search(v, visitados):
    """Visita v y luego, de forma recursiva, cada uno de sus hijos."""
    visitados.append(v)                      # visit v
    for u in arbol.get(v, []):               # for each child u of v
        depth_first_search(u, visitados)

orden = []
depth_first_search(1, orden)
print("Orden de visita DFS:", orden)

Orden de visita DFS: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


**Clave:** el orden de visita coincide con la numeración del árbol del PDF (1, 2, 3, ..., 12). La DFS se escribe con recursión y no necesita una estructura de datos adicional: la pila de llamadas hace de pila.

**Ampliación (no está en el PDF):** para ver cómo trabaja la pila de llamadas, el siguiente ejemplo imprime una línea al entrar a cada nodo y otra al volver (retroceder) de él, con sangría proporcional a la profundidad.

In [3]:
def dfs_con_traza(v, nivel=0):
    print("  " * nivel + f"entrar a {v}")
    for u in arbol.get(v, []):
        dfs_con_traza(u, nivel + 1)
    print("  " * nivel + f"volver de {v}")

dfs_con_traza(2)

entrar a 2
  entrar a 3
  volver de 3
  entrar a 4
  volver de 4
  entrar a 5
  volver de 5
volver de 2


**Referencia:** PDF parte 1, págs. 49–54

## Estrategia de retroceso, prometedor y poda

El **retroceso** (*backtracking*) es una estrategia de diseño de algoritmos que construye de forma incremental una solución candidata y retrocede en cuanto determina que esa candidata no puede conducir a una solución válida. Resuelve los problemas con este proceso:

1. Organizar el espacio de búsqueda en forma de árbol (el **árbol del espacio de estados**).
2. Explorar ese árbol con búsqueda en profundidad.
3. Volver atrás cuando el nodo ya no es prometedor.

La técnica de **poda** (*pruning*) consiste en volver sin visitar los nodos del subárbol que no son prometedores. **Prometedor** (*promising*) se refiere a un estado en el que todavía puede existir una solución en el subárbol del nodo visitado.

En el ejemplo del PDF, si los nodos 5, 6 y 10 no son prometedores, no hace falta visitarlos ni visitar sus subnodos. Podándolos, el algoritmo evita esas visitas y mejora su eficiencia: solo se recorren 1, 2, 3, 4, 9, 11 y 12.

**Referencia:** PDF parte 1, págs. 55–57

In [4]:
# Mismo árbol, ahora con poda: no se visitan los nodos no prometedores (5, 6 y 10 en el PDF)
no_prometedores = {5, 6, 10}

def dfs_con_poda(v, visitados):
    if v in no_prometedores:                 # poda: se vuelve sin visitar el subárbol
        return
    visitados.append(v)
    for u in arbol.get(v, []):
        dfs_con_poda(u, visitados)

orden_podado = []
dfs_con_poda(1, orden_podado)
print("Orden con poda:", orden_podado)
print("Nodos ahorrados:", sorted(set(range(1, 13)) - set(orden_podado)))

Orden con poda: [1, 2, 3, 4, 9, 11, 12]
Nodos ahorrados: [5, 6, 7, 8, 10]


**Clave:** podar no cambia la corrección si la prueba de "prometedor" es correcta, solo evita trabajo inútil. Con 6 podado se ahorran también 7 y 8; con 5 y 10 solo esos nodos (que no tienen hijos en el árbol dibujado).

**Error común:** confundir "no prometedor" con "no solución". Un nodo no prometedor garantiza que no hay solución en su subárbol; un nodo prometedor puede aún no ser solución (hay que seguir bajando).

**Referencia:** PDF parte 1, págs. 55–57

## El problema de las N reinas con retroceso

El PDF razona con N=4 (cuatro reinas en un tablero de 4*4). La idea más simple es colocar una reina por fila e ir probando todas las columnas posibles; en cuanto una posición choca con las reinas ya colocadas, se descarta y se prueba otra. Si ninguna columna sirve en una fila, se retrocede a la fila anterior y se mueve su reina.

Recorrido del PDF para N=4 (filas y columnas desde 0):

1. Reina en (0, 0). En la fila 1 las columnas 0 y 1 no sirven (misma columna y diagonal); se coloca en la columna 2.
2. En la fila 2 no hay ninguna columna posible (comparte columna, fila o diagonal con las dos reinas colocadas): se retrocede.
3. Se vuelve a la fila 1 y se prueba la columna 3. Ahora la fila 2 admite la columna 1, pero la fila 3 ya no admite ninguna columna.
4. Se retrocede a la fila 2 (sin más opciones), luego a la fila 1 (ya probó todo) y por último a la fila 0, donde se prueba la columna 1.
5. Con la reina en (0, 1): fila 1 en la columna 3, fila 2 en la columna 0 y fila 3 en la columna 2. Hay cuatro reinas: problema resuelto.

**Referencia:** PDF parte 1, págs. 59–65

### Árbol del espacio de estados

El PDF dibuja el árbol de búsqueda del problema con N=4 (pág. 66). Cada nodo es un par (fila, columna) numerado desde 1, y `S` es la raíz (tablero vacío). Los nodos marcados con una X no son prometedores y se podan. De las cuatro columnas posibles en la fila 1, la rama (1, 1) lleva a callejones sin salida en la fila 3 o 4, y la rama (1, 2) llega a la solución (1,2), (2,4), (3,1), (4,3), que en índices desde 0 es `col = [1, 3, 0, 2]`.

El pseudocódigo del PDF (pág. 67) es exactamente el esquema general del retroceso: si el nodo es prometedor, se comprueba si es solución y, si no lo es, se visita a cada hijo.

```
def n_queens(node v):
    if (promising(v)):
        if (there is a solution at v):
            write the solution;
        else
            for (each child u of v):
                n_queens(u);
```

**Referencia:** PDF parte 1, págs. 66–67

### Representación y función `promising`

Se representa el estado con una lista `col` donde `col[i]` es el número de la columna de la reina de la fila `i`. Por ejemplo `col = [1, 0, 2, 3]` significa las posiciones (0, 1), (1, 0), (2, 2) y (3, 3).

Como solo se coloca una reina por fila, nunca coinciden dos en la misma fila. La función prometedora comprueba las otras dos condiciones entre las reinas `i` y `k`:

- misma columna: `col[i] == col[k]`;
- misma diagonal: `abs(col[i] - col[k]) == i - k` (la diferencia de columnas es igual a la diferencia de filas).

**Nota sobre el PDF:** las diapositivas de la pág. 69 y 70 escriben las condiciones con el índice `j` (`col[i] == col[j]`, `|col[i] - col[j]| == i - j`), pero el código final usa `k`. Aquí se usa `k`, como en el código. Los tableros dibujados en esas páginas están numerados de 1 a 8, mientras que `col` usa índices desde 0.

In [5]:
def promising(i, col):
    """Indica si la reina de la fila i es compatible con las de las filas 0 a i - 1."""
    for k in range(i):                                   # filas ya colocadas: 0 a i - 1
        if col[i] == col[k] or abs(col[i] - col[k]) == (i - k):
            return False                                 # misma columna o misma diagonal
    return True

# Estado del PDF (pág. 68): col = [1, 0, 2, 3]
col = [1, 0, 2, 3]
print("Posiciones:", [(fila, col[fila]) for fila in range(len(col))])
for i in range(len(col)):
    print(f"promising({i}, col) =", promising(i, col))

Posiciones: [(0, 1), (1, 0), (2, 2), (3, 3)]
promising(0, col) = True
promising(1, col) = False
promising(2, col) = True
promising(3, col) = False


En `col = [1, 0, 2, 3]` la fila 1 (columna 0) es incompatible con la fila 0 (columna 1) porque están en una diagonal: |0 − 1| = 1 = 1 − 0. Por eso `promising(1, col)` devuelve `False`.

**Referencia:** PDF parte 1, págs. 68–72 (Líneas 1–2: recibe la lista y el índice y recorre las filas 0 a i-1; Líneas 3–5: compara columna y diagonal y devuelve `False` si hay choque, `True` en caso contrario).

### La función `n_queens`

`n_queens(i, col)` implementa el retroceso:

- **Líneas 2–4:** si `promising(i, col)` es falso, la llamada termina de inmediato (poda). Si es verdadero y `i` es la última fila (`i == len(col) - 1`), se encontró una solución y se imprime `col`.
- **Líneas 5–8:** si es prometedor pero no es la última fila, se sigue la búsqueda en profundidad: para cada columna `j` se coloca la reina de la fila `i + 1` en `j` (`col[i + 1] = j`) y se llama a `n_queens(i + 1, col)`.

La primera llamada es `n_queens(-1, [-1] * N)`: se parte de la fila `-1` (aún no hay reinas) con una lista de N valores `-1`. Con `i = -1`, `promising` no recorre nada (`range(-1)` está vacío) y devuelve `True`, así que se expande la fila 0.

**Nota sobre el PDF:** al volver de una rama, `col[i + 1]` conserva el valor de la prueba anterior. No hace falta borrarlo, porque el siguiente `j` lo sobrescribe y `promising` solo mira las filas hasta `i`.

**Referencia:** PDF parte 1, págs. 73–75

In [6]:
def n_queens(i, col):
    """Retroceso para N reinas: imprime cada solución (lista col) que encuentra."""
    if promising(i, col):
        if i == len(col) - 1:                  # última fila: solución completa
            print(col)
        else:
            for j in range(len(col)):          # probar cada columna en la fila siguiente
                col[i + 1] = j
                n_queens(i + 1, col)

# Ejecución del PDF con N = 4 (en el PDF: N = int(input("Input the number of queens: ")))
N = 4
n_queens(-1, [-1] * N)

[1, 3, 0, 2]
[2, 0, 3, 1]


Salida esperada (pág. 75 del PDF): `[1, 3, 0, 2]` y `[2, 0, 3, 1]`, las dos únicas soluciones para N=4. Coincide con la ejecución anterior.

**Clave:** el retroceso aparece de forma natural: cuando `promising` devuelve `False`, la llamada termina y el control vuelve al `for` de la llamada anterior, que prueba la siguiente columna. No hay código explícito de "deshacer".

**Error común:** olvidar la condición de fila final y seguir expandiendo, o pasar `col` sin inicializar con `-1` (los valores `-1` en filas aún no colocadas nunca se comparan, porque `promising` solo mira hasta la fila `i`).

**Para el quiz:** `n_queens(-1, [-1] * N)` arranca la búsqueda; `promising` comprueba columna y diagonal con `col[i] == col[k]` y `abs(col[i] - col[k]) == i - k`.

**Referencia:** PDF parte 1, págs. 73–75

## Proyecto de la unidad

**Problema:** colocar N reinas en un tablero de N*N de modo que ninguna comparta fila, columna ni diagonal con otra (PDF págs. 44–47).

**Plan de programación (los pasos del PDF):**

1. Representar el estado con la lista `col`, donde `col[i]` es la columna de la reina de la fila `i`.
2. Escribir `promising(i, col)` para detectar choques de columna y de diagonal.
3. Escribir `n_queens(i, col)` con el esquema de retroceso: podar si no es prometedor, imprimir si es la última fila, expandir la siguiente fila si no.
4. Llamar `n_queens(-1, [-1] * N)` con el N elegido por el usuario.

El código final del PDF (pág. 47) es el que ya se definió arriba. A continuación se encapsula la entrada del usuario en una función; la versión no interactiva se ejecuta con una lista de valores de prueba.

In [7]:
def ejecutar_n_reinas():
    """Versión interactiva del PDF: pide N y resuelve el problema."""
    N = int(input("Input the number of queens: "))
    n_queens(-1, [-1] * N)

# Para usarlo de forma interactiva, quita el comentario de la siguiente línea
# ejecutar_n_reinas()

# Versión no interactiva: valores de prueba en lugar de input()
for N in [1, 4, 5]:
    print(f"N = {N}")
    n_queens(-1, [-1] * N)

N = 1
[0]
N = 4
[1, 3, 0, 2]
[2, 0, 3, 1]
N = 5
[0, 2, 4, 1, 3]
[0, 3, 1, 4, 2]
[1, 3, 0, 2, 4]
[1, 4, 2, 0, 3]
[2, 0, 3, 1, 4]
[2, 4, 1, 3, 0]
[3, 0, 2, 4, 1]
[3, 1, 4, 2, 0]
[4, 1, 3, 0, 2]
[4, 2, 0, 3, 1]


**Ampliación (no está en el PDF):** el PDF solo imprime las soluciones como listas. Las siguientes funciones recogen las soluciones en una lista (para poder contarlas) y dibujan el tablero en texto, con `R` en la reina y `.` en las casillas vacías.

In [8]:
def soluciones_n_reinas(N):
    """Devuelve la lista de todas las soluciones (cada una como lista col)."""
    resultado = []
    def buscar(i, col):
        if promising(i, col):
            if i == len(col) - 1:
                resultado.append(col.copy())   # copia: col se sigue modificando
            else:
                for j in range(len(col)):
                    col[i + 1] = j
                    buscar(i + 1, col)
    buscar(-1, [-1] * N)
    return resultado

def mostrar_tablero(col):
    for fila in range(len(col)):
        print(" ".join("R" if c == col[fila] else "." for c in range(len(col))))

for sol in soluciones_n_reinas(4):
    print(sol)
    mostrar_tablero(sol)
    print()

print("Número de soluciones para N = 1 a 8:",
      [len(soluciones_n_reinas(n)) for n in range(1, 9)])

[1, 3, 0, 2]
. R . .
. . . R
R . . .
. . R .

[2, 0, 3, 1]
. . R .
R . . .
. . . R
. R . .

Número de soluciones para N = 1 a 8: [1, 0, 0, 2, 10, 4, 40, 92]


**Clave:** para N=8 existen 92 soluciones; para N=2 y N=3 no hay ninguna.

**Ampliación (no está en el PDF):** el beneficio de la poda se ve al contar nodos. Sin poda habría que examinar todos los tableros con una reina por fila (N^N). Con poda solo se visitan los prefijos compatibles.

In [9]:
def nodos_visitados(N):
    """Cuenta las llamadas a n_queens (nodos del árbol) con poda."""
    cuenta = 0
    def buscar(i, col):
        nonlocal cuenta
        cuenta += 1
        if promising(i, col) and i < len(col) - 1:
            for j in range(len(col)):
                col[i + 1] = j
                buscar(i + 1, col)
    buscar(-1, [-1] * N)
    return cuenta

for n in [4, 6, 8]:
    print(f"N = {n}: nodos con poda = {nodos_visitados(n)}, tableros sin poda = {n ** n}")

N = 4: nodos con poda = 61, tableros sin poda = 256
N = 6: nodos con poda = 895, tableros sin poda = 46656
N = 8: nodos con poda = 15721, tableros sin poda = 16777216


**Referencia:** PDF parte 1, págs. 44–47 y 59–75

## Prueba sorpresa

**Enunciado (PDF págs. 77–78):** se sustituye cada letra de `A MERRY XMAS TO ALL` por un dígito (letras distintas, dígitos distintos). El número formado por cada una de las cinco palabras debe ser un cuadrado perfecto y, además, la suma de los dígitos de cada palabra también debe ser un cuadrado perfecto. Hay que hallar el dígito de cada letra.

**Sugerencia del PDF:** T y O pueden ser 8 y 1. Como 81 es un cuadrado perfecto (81 = 9²) y 8 + 1 = 9 = 3² también, la palabra `TO` cumple ambas condiciones.

**Resolución con retroceso (ampliación):** se asigna un dígito a cada letra en orden. En cuanto todas las letras de una palabra tienen dígito, se comprueba esa palabra y, si no cumple, se poda la rama. Se prohíbe el cero como primer dígito de una palabra.

In [10]:
from math import isqrt

def es_cuadrado(n):
    return isqrt(n) ** 2 == n

palabras = ["A", "MERRY", "XMAS", "TO", "ALL"]
letras = list("ALTOXMSERY")                      # orden de asignación
iniciales = {p[0] for p in palabras}

def valor(palabra, asignacion):
    return int("".join(str(asignacion[c]) for c in palabra))

def cumple(asignacion):
    """Comprueba las palabras cuyas letras ya están asignadas (poda)."""
    for p in palabras:
        if all(c in asignacion for c in p):
            n = valor(p, asignacion)
            if not es_cuadrado(n) or not es_cuadrado(sum(int(d) for d in str(n))):
                return False
    return True

soluciones_puzzle = []

def asignar(i, asignacion, usados):
    if not cumple(asignacion):                   # no prometedor: retroceder
        return
    if i == len(letras):
        soluciones_puzzle.append(dict(asignacion))
        return
    letra = letras[i]
    for d in range(10):
        if d in usados or (d == 0 and letra in iniciales):
            continue
        asignacion[letra] = d
        usados.add(d)
        asignar(i + 1, asignacion, usados)
        usados.discard(d)                        # deshacer antes de probar otro dígito
        del asignacion[letra]

asignar(0, {}, set())
print("Soluciones encontradas:", len(soluciones_puzzle))
sol = soluciones_puzzle[0]
print(sol)
for p in palabras:
    n = valor(p, sol)
    print(f"{p} = {n} = {isqrt(n)}^2, suma de dígitos = {sum(int(d) for d in str(n))}")

Soluciones encontradas: 1
{'A': 9, 'L': 0, 'T': 8, 'O': 1, 'X': 7, 'M': 3, 'S': 6, 'E': 4, 'R': 2, 'Y': 5}
A = 9 = 3^2, suma de dígitos = 9
MERRY = 34225 = 185^2, suma de dígitos = 16
XMAS = 7396 = 86^2, suma de dígitos = 25
TO = 81 = 9^2, suma de dígitos = 9
ALL = 900 = 30^2, suma de dígitos = 9


Solución única: A=9, M=3, E=4, R=2, Y=5, X=7, S=6, T=8, O=1, L=0. Así `A` = 9, `MERRY` = 34225 (185²), `XMAS` = 7396 (86²), `TO` = 81 (9²) y `ALL` = 900 (30²), y la suma de dígitos de cada palabra (9, 16, 25, 9 y 9) también es un cuadrado perfecto. Coincide con la sugerencia del PDF (T=8, O=1).

**Nota sobre el PDF:** en la página 77 la tabla de letras se ve parcialmente distorsionada al renderizarla, pero el enunciado y la sugerencia de la pág. 78 se leen completos en el texto extraído. En el PDF solo están dados T=8 y O=1.

**Referencia:** PDF parte 1, págs. 77–78

## Programación en pareja

Las págs. 80–81 del PDF son material pedagógico sobre la práctica de programación en pareja (bilingüe, con el texto en inglés y su traducción superpuestos). Ideas principales:

- **Pautas, mecanismos y plan de contingencia:** los alumnos deben turnarse para manejar el mouse, y hay que tener un plan si uno de los socios falta o decide no participar, aclarando que el estudiante activo no será penalizado.
- **Emparejar con habilidades similares**, no necesariamente iguales: el emparejamiento desigual suele producir una participación desbalanceada. La programación en pareja no es una estrategia de "divide y vencerás", sino un esfuerzo de colaboración real en todo el proyecto.
- **Motivar con incentivos adicionales:** a algunos docentes les resulta útil exigir la pareja solo en una o dos tareas.
- **Evitar trampas en la colaboración:** revisar el diseño y la evaluación del curso, y hablar de forma explícita sobre las conductas que se considerarán trampa; hacer las tareas significativas.
- **Entorno de aprendizaje colaborativo:** actividades formales o informales en las que los estudiantes trabajan juntos y conversan sobre los problemas.

**Referencia:** PDF parte 1, págs. 79–81

### Ejercicio de pareja: laberinto N*N

**Enunciado (págs. 82–84):** una matriz binaria N*N representa un laberinto. El ratón debe moverse desde la coordenada (0, 0) hasta la (N-1, N-1) y solo puede pasar por las casillas con valor 0. Hay que escribir un programa que muestre la ruta de escape.

**Sugerencias del PDF:** viajar solo por casillas con 0; continuar con DFS y retroceder al llegar a un callejón sin salida, hasta alcanzar el destino.

El PDF muestra solo la llamada y su salida para un laberinto de 4*4 (pág. 84): la ruta se imprime como una matriz en la que las casillas del camino valen 0 y las demás 1. No incluye el código de `solve`, por lo que la siguiente implementación es propia.

**Ampliación (no está en el PDF):** la implementación de `solve` con retroceso. Se marca cada casilla visitada para no volver a ella; si desde una casilla no hay salida, se desmarca como parte del camino (retroceso) pero se conserva como visitada para no repetir trabajo.

In [11]:
def buscar_camino(laberinto):
    """Devuelve la ruta [(fila, col), ...] de (0, 0) a (N-1, N-1) por casillas con 0, o None."""
    n = len(laberinto)
    if laberinto[0][0] != 0 or laberinto[n - 1][n - 1] != 0:
        return None
    visitado = [[False] * n for _ in range(n)]
    camino = []

    def dfs(f, c):
        if not (0 <= f < n and 0 <= c < n):      # fuera del tablero
            return False
        if laberinto[f][c] != 0 or visitado[f][c]:   # pared o ya visitada
            return False
        visitado[f][c] = True
        camino.append((f, c))
        if (f, c) == (n - 1, n - 1):             # destino alcanzado
            return True
        for df, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:   # derecha, abajo, izquierda, arriba
            if dfs(f + df, c + dc):
                return True
        camino.pop()                             # callejón sin salida: retroceder
        return False

    return camino if dfs(0, 0) else None

def solve(laberinto):
    """Imprime la ruta como en el PDF: 0 en el camino y 1 en el resto."""
    camino = buscar_camino(laberinto)
    if camino is None:
        print("No hay ruta de escape")
        return
    ruta = set(camino)
    for f in range(len(laberinto)):
        print(" ".join("0" if (f, c) in ruta else "1" for c in range(len(laberinto))))

# Ejemplo de la pág. 84 del PDF
maze = [[0, 0, 0, 0],
        [1, 0, 1, 0],
        [1, 0, 1, 1],
        [0, 0, 0, 0]]
solve(maze)

0 0 1 1
1 0 1 1
1 0 1 1
1 0 0 0


La salida coincide con la del PDF (`0 0 1 1 / 1 0 1 1 / 1 0 1 1 / 1 0 0 0`). Ahora el laberinto de 10*10 de la pág. 82. Para distinguir las casillas se dibuja el tablero con `#` en las paredes (1), `.` en los pasillos libres (0) y `o` en la ruta de escape.

**Nota sobre el PDF:** la pág. 83 dibuja el laberinto junto a un árbol de decisión con un ratón en la raíz: las hojas marcadas con `x` son callejones sin salida y la marcada con `o` es la salida (el texto extraído de esa página solo recoge esas marcas). El PDF no muestra la ruta completa de este laberinto, así que el resultado de abajo se calculó con la implementación propia.

In [12]:
maze10 = [
    [0, 0, 1, 1, 0, 0, 0, 0, 1, 1],
    [1, 0, 0, 0, 0, 1, 1, 0, 1, 1],
    [1, 1, 1, 0, 1, 1, 1, 0, 0, 0],
    [1, 0, 0, 0, 0, 1, 0, 0, 1, 1],
    [1, 0, 1, 1, 0, 1, 1, 1, 1, 1],
    [1, 0, 1, 1, 0, 1, 1, 0, 0, 0],
    [1, 0, 1, 1, 0, 0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1, 1, 1, 0, 0, 1],
    [1, 1, 0, 0, 1, 1, 1, 1, 0, 1],
    [1, 1, 1, 0, 1, 1, 1, 1, 0, 0],
]

camino10 = buscar_camino(maze10)
print("Longitud de la ruta:", None if camino10 is None else len(camino10))
if camino10:
    ruta = set(camino10)
    for f in range(10):
        print(" ".join("o" if (f, c) in ruta else ("#" if maze10[f][c] else ".") for c in range(10)))

Longitud de la ruta: 19
o o # # . . . . # #
# o o o . # # . # #
# # # o # # # . . .
# . . o o # . . # #
# . # # o # # # # #
# . # # o # # . . .
# . # # o o o o # #
. . . # # # # o o #
# # . . # # # # o #
# # # . # # # # o o


**Referencia:** PDF parte 1, págs. 82–84

## Práctica

Siete ejercicios de dificultad creciente. Completa cada función y ejecuta la celda de verificación; mientras no la resuelvas, imprimirá "Incorrecto".

### Ejercicio 1 — Recorrido DFS de un árbol

Escribe `dfs_preorden(arbol, nodo)` que devuelva la lista de nodos en el orden en que los visita una DFS que parte de `nodo`. `arbol` es un diccionario nodo -> lista de hijos (los nodos sin hijos pueden no aparecer).

In [13]:
def dfs_preorden(arbol, nodo):
    # Escribe tu solución aquí
    pass

In [14]:
arbol_pdf = {1: [2, 6, 9], 2: [3, 4, 5], 6: [7, 8], 9: [10, 11, 12]}
verificar("árbol del PDF", dfs_preorden(arbol_pdf, 1), [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])
verificar("desde el nodo 6", dfs_preorden(arbol_pdf, 6), [6, 7, 8])
verificar("hoja", dfs_preorden(arbol_pdf, 12), [12])

Incorrecto: árbol del PDF. Se esperaba [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] y se obtuvo None
Incorrecto: desde el nodo 6. Se esperaba [6, 7, 8] y se obtuvo None
Incorrecto: hoja. Se esperaba [12] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def dfs_preorden(arbol, nodo):
    orden = [nodo]
    for hijo in arbol.get(nodo, []):
        orden += dfs_preorden(arbol, hijo)
    return orden
```

Se visita el nodo y luego, de forma recursiva, cada hijo en orden. La pila de llamadas guarda el camino.
</details>

### Ejercicio 2 — La función promising

Escribe `promising(i, col)` que devuelva `True` si la reina de la fila `i` no comparte columna ni diagonal con las reinas de las filas `0` a `i - 1`.

In [15]:
def promising(i, col):
    # Escribe tu solución aquí
    pass

In [16]:
verificar("diagonal", promising(1, [0, 1, -1, -1]), False)
verificar("compatible", promising(1, [0, 2, -1, -1]), True)
verificar("misma columna", promising(2, [1, 3, 1, -1]), False)
verificar("sin reinas previas", promising(0, [2, -1, -1, -1]), True)

Incorrecto: diagonal. Se esperaba False y se obtuvo None
Incorrecto: compatible. Se esperaba True y se obtuvo None
Incorrecto: misma columna. Se esperaba False y se obtuvo None
Incorrecto: sin reinas previas. Se esperaba True y se obtuvo None


<details><summary>Ver solución</summary>

```python
def promising(i, col):
    for k in range(i):
        if col[i] == col[k] or abs(col[i] - col[k]) == (i - k):
            return False
    return True
```

Se compara la fila `i` con cada fila anterior `k`: la misma columna o una diferencia de columnas igual a la diferencia de filas indica ataque.
</details>

### Ejercicio 3 — Contar soluciones de N reinas

Escribe `contar_n_reinas(n)` que devuelva cuántas soluciones tiene el problema de las N reinas. Puedes usar `promising` (ya definida arriba).

In [17]:
def contar_n_reinas(n):
    # Escribe tu solución aquí
    pass

In [18]:
verificar("N = 4", contar_n_reinas(4), 2)
verificar("N = 6", contar_n_reinas(6), 4)
verificar("N = 8", contar_n_reinas(8), 92)
verificar("N = 3", contar_n_reinas(3), 0)

Incorrecto: N = 4. Se esperaba 2 y se obtuvo None
Incorrecto: N = 6. Se esperaba 4 y se obtuvo None
Incorrecto: N = 8. Se esperaba 92 y se obtuvo None
Incorrecto: N = 3. Se esperaba 0 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def contar_n_reinas(n):
    cuenta = 0
    def buscar(i, col):
        nonlocal cuenta
        if promising(i, col):
            if i == n - 1:
                cuenta += 1
            else:
                for j in range(n):
                    col[i + 1] = j
                    buscar(i + 1, col)
    buscar(-1, [-1] * n)
    return cuenta
```

Es el mismo esquema de `n_queens`, pero en lugar de imprimir se incrementa un contador.
</details>

### Ejercicio 4 — Permutaciones con retroceso

Escribe `permutaciones(items)` que devuelva todas las permutaciones de una lista, como listas, en el orden en que las genera una DFS que elige los elementos de izquierda a derecha.

In [19]:
def permutaciones(items):
    # Escribe tu solución aquí
    pass

In [20]:
verificar("tres elementos", permutaciones([1, 2, 3]),
          [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]])
verificar("un elemento", permutaciones(["a"]), [["a"]])
verificar("cuatro elementos", len(permutaciones([1, 2, 3, 4]) or []), 24)

Incorrecto: tres elementos. Se esperaba [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]] y se obtuvo None
Incorrecto: un elemento. Se esperaba [['a']] y se obtuvo None
Incorrecto: cuatro elementos. Se esperaba 24 y se obtuvo 0


<details><summary>Ver solución</summary>

```python
def permutaciones(items):
    resultado = []
    usado = [False] * len(items)
    actual = []
    def buscar():
        if len(actual) == len(items):
            resultado.append(actual.copy())
            return
        for i in range(len(items)):
            if not usado[i]:
                usado[i] = True
                actual.append(items[i])
                buscar()
                actual.pop()           # retroceso: deshacer la elección
                usado[i] = False
    buscar()
    return resultado
```

Se elige un elemento no usado, se explora y luego se deshace la elección (`pop` y desmarcar). Ese deshacer es el retroceso.
</details>

### Ejercicio 5 — Suma de subconjuntos con poda

Escribe `subconjuntos_suma(numeros, objetivo)` que devuelva todos los subconjuntos (como listas, conservando el orden de `numeros`) cuya suma sea exactamente `objetivo`. Todos los números son positivos: poda cuando la suma parcial supere el objetivo.

In [21]:
def subconjuntos_suma(numeros, objetivo):
    # Escribe tu solución aquí
    pass

In [22]:
verificar("objetivo 9", subconjuntos_suma([3, 4, 5, 6], 9), [[3, 6], [4, 5]])
verificar("objetivo 6", subconjuntos_suma([2, 4, 6], 6), [[2, 4], [6]])
verificar("sin solución", subconjuntos_suma([5, 10], 3), [])

Incorrecto: objetivo 9. Se esperaba [[3, 6], [4, 5]] y se obtuvo None
Incorrecto: objetivo 6. Se esperaba [[2, 4], [6]] y se obtuvo None
Incorrecto: sin solución. Se esperaba [] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def subconjuntos_suma(numeros, objetivo):
    resultado = []
    def buscar(inicio, actual, suma):
        if suma == objetivo:
            resultado.append(actual.copy())
            return
        if suma > objetivo:                      # poda: ya no es prometedor
            return
        for i in range(inicio, len(numeros)):
            actual.append(numeros[i])
            buscar(i + 1, actual, suma + numeros[i])
            actual.pop()                         # retroceso
    buscar(0, [], 0)
    return resultado
```

Cada nodo es un subconjunto parcial. Si la suma lo iguala se guarda; si lo supera, se poda porque añadir números positivos solo la aumentaría.
</details>

### Ejercicio 6 — Camino en un laberinto

Escribe `camino_laberinto(laberinto)` que devuelva la lista de coordenadas `(fila, columna)` desde `(0, 0)` hasta `(N-1, N-1)` pasando solo por casillas con 0, o `None` si no hay ruta. Solo se permiten movimientos arriba, abajo, izquierda y derecha.

In [23]:
def camino_laberinto(laberinto):
    # Escribe tu solución aquí
    pass

In [24]:
m1 = [[0, 0, 0, 0], [1, 0, 1, 0], [1, 0, 1, 1], [0, 0, 0, 0]]
verificar("laberinto del PDF", camino_laberinto(m1),
          [(0, 0), (0, 1), (1, 1), (2, 1), (3, 1), (3, 2), (3, 3)])
verificar("sin salida", camino_laberinto([[0, 1], [1, 0]]), None)
verificar("un solo casillero", camino_laberinto([[0]]), [(0, 0)])

Incorrecto: laberinto del PDF. Se esperaba [(0, 0), (0, 1), (1, 1), (2, 1), (3, 1), (3, 2), (3, 3)] y se obtuvo None
Correcto: sin salida
Incorrecto: un solo casillero. Se esperaba [(0, 0)] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def camino_laberinto(laberinto):
    n = len(laberinto)
    visitado = [[False] * n for _ in range(n)]
    camino = []
    def dfs(f, c):
        if not (0 <= f < n and 0 <= c < n) or laberinto[f][c] != 0 or visitado[f][c]:
            return False
        visitado[f][c] = True
        camino.append((f, c))
        if (f, c) == (n - 1, n - 1):
            return True
        for df, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
            if dfs(f + df, c + dc):
                return True
        camino.pop()                             # callejón sin salida: retroceder
        return False
    return camino if dfs(0, 0) else None
```

Se marcan las casillas visitadas para evitar ciclos. Si ninguna dirección lleva al destino, se quita la casilla del camino y se retrocede.
</details>

### Ejercicio 7 — Paréntesis balanceados

Escribe `parentesis(n)` que devuelva todas las cadenas de `n` pares de paréntesis bien balanceadas. Poda: no abras más de `n` paréntesis ni cierres uno sin apertura pendiente. Devuelve la lista (el orden no importa).

In [25]:
def parentesis(n):
    # Escribe tu solución aquí
    pass

In [26]:
verificar("n = 1", sorted(parentesis(1) or []), ["()"])
verificar("n = 2", sorted(parentesis(2) or []), ["(())", "()()"])
verificar("n = 3", sorted(parentesis(3) or []), ["((()))", "(()())", "(())()", "()(())", "()()()"])

Incorrecto: n = 1. Se esperaba ['()'] y se obtuvo []
Incorrecto: n = 2. Se esperaba ['(())', '()()'] y se obtuvo []
Incorrecto: n = 3. Se esperaba ['((()))', '(()())', '(())()', '()(())', '()()()'] y se obtuvo []


<details><summary>Ver solución</summary>

```python
def parentesis(n):
    resultado = []
    def buscar(actual, abiertos, cerrados):
        if len(actual) == 2 * n:
            resultado.append(actual)
            return
        if abiertos < n:                         # aún se puede abrir
            buscar(actual + "(", abiertos + 1, cerrados)
        if cerrados < abiertos:                  # solo se cierra si hay una apertura pendiente
            buscar(actual + ")", abiertos, cerrados + 1)
    buscar("", 0, 0)
    return resultado
```

Las dos condiciones de poda impiden construir cadenas inválidas, así que cada hoja alcanzada es una solución.
</details>

## Apuntes para estudiar

**Clave**
- Retroceso = búsqueda en profundidad sobre el árbol del espacio de estados, construyendo la solución de forma incremental y volviendo atrás cuando un candidato no puede conducir a una solución válida.
- La búsqueda en profundidad se implementa con recursión: la pila de llamadas guarda el camino.
- Poda: volver sin visitar el subárbol de un nodo no prometedor. Prometedor: aún puede haber solución en su subárbol.
- Esquema: `if promising(v): if hay_solucion(v): escribir; else: for cada hijo u de v: n_queens(u)`.
- N reinas: `col[i]` es la columna de la reina de la fila `i`; choque si `col[i] == col[k]` o `abs(col[i] - col[k]) == i - k`.
- Llamada inicial: `n_queens(-1, [-1] * N)`. Para N=4 las soluciones son `[1, 3, 0, 2]` y `[2, 0, 3, 1]`.

**Errores comunes**
- Olvidar comprobar la última fila y no detener la recursión.
- No deshacer la elección al volver (en estructuras como listas o conjuntos usados) cuando el estado se comparte entre ramas.
- Guardar `col` sin copiarla al almacenar soluciones: la lista se sigue modificando.
- Confundir fila y columna, o usar una diferencia de columnas sin valor absoluto.
- Tomar "prometedor" como "solución": un nodo prometedor aún puede no tener solución.

**Para el quiz**
- El retroceso es una DFS con poda; la poda mejora la eficiencia sin perder soluciones.
- Condiciones de choque entre reinas: misma columna y valor absoluto de la diferencia de columnas igual a la diferencia de filas.
- Palabras clave: retroceso, prometedor, búsqueda en profundidad, poda.
- El primer valor de `n_queens` es `-1` porque aún no hay ninguna fila colocada.
- La unidad se rotula "UNIDAD 32" en las diapositivas de objetivos del PDF, pero es la Unidad 33.

## Autoevaluación

**1. ¿Qué es el retroceso y en qué se diferencia de una búsqueda exhaustiva?**

<details><summary>Ver respuesta</summary>

Es una búsqueda en profundidad que construye la solución de forma incremental y vuelve atrás cuando el candidato parcial no puede llevar a una solución válida. La búsqueda exhaustiva examina todos los candidatos completos; el retroceso descarta ramas enteras al detectar el fallo en un prefijo (poda).
</details>

**2. ¿Qué significa que un nodo sea "prometedor"?**

<details><summary>Ver respuesta</summary>

Que en el subárbol del nodo visitado todavía puede existir una solución. Si no es prometedor, se poda: se vuelve sin visitar sus descendientes.
</details>

**3. En N reinas, ¿cómo se detecta que dos reinas se atacan en diagonal?**

<details><summary>Ver respuesta</summary>

Si el valor absoluto de la diferencia de columnas es igual a la diferencia de filas: `abs(col[i] - col[k]) == i - k`. La misma columna se detecta con `col[i] == col[k]`; la misma fila no ocurre porque se coloca una reina por fila.
</details>

**4. ¿Por qué se llama `n_queens(-1, [-1] * N)` al empezar?**

<details><summary>Ver respuesta</summary>

Porque se parte de un tablero sin reinas: la fila `-1` es la raíz del árbol. `promising(-1, col)` no recorre ninguna fila y devuelve `True`, y la rama `else` expande la fila 0. La lista de `-1` indica que ninguna fila tiene reina todavía.
</details>

**5. ¿Dónde está el "retroceso" en el código de `n_queens`?**

<details><summary>Ver respuesta</summary>

En la recursión: cuando `promising` devuelve `False` la llamada termina (`return` implícito) y el control vuelve al `for` de la llamada anterior, que prueba el siguiente `j`. La pila de llamadas hace de memoria del camino, por lo que no hay código explícito para deshacer.
</details>

[Índice de la semana](../semana-05.ipynb) | [Anterior](u32-programacion-dinamica.ipynb) | [Siguiente](../clase-2-modulos-y-pandas/u34-modulos-de-python.ipynb)